# `ui/main.py` — API Validation Walkthrough

Exercises the FastAPI backend using `TestClient`, which runs the app in-process
without needing to start a real server.  The same client is used by the unit tests;
here we focus on the full request/response flow with real data.

**Endpoints tested:**
- `POST /ask` — entry point; returns a plan or a disambiguation prompt
- `POST /plan` — used after disambiguation to scope the plan to a specific section
- `GET /images/{path}` — serves diagram images from `data/images/`

In [ ]:
import sys, json, os
sys.path.insert(0, "..")
os.chdir("..")  # resolve data/chroma and data/images relative to project root

from dotenv import load_dotenv
load_dotenv(".env")

# TestClient runs the app in-process — no uvicorn needed
from fastapi.testclient import TestClient
from ui.main import app

client = TestClient(app)
print(f"Working directory: {os.getcwd()}")
print("TestClient ready")

## 1. Unambiguous query → plan returned directly

A query that matches only one chapter goes straight through retrieval and planning
without a disambiguation step.  The response `type` is `"plan"`.

"Disassemble the fork cartridge" is cleanly contained in Ch. 6 — no cross-chapter
bleed — and also exercises the prerequisite walk (6.10 and 6.11 surface as depth-1
prerequisites of 6.12).

In [ ]:
resp = client.post("/ask", json={"query": "disassemble the fork cartridge"})
print(f"Status: {resp.status_code}")

body = resp.json()
print(f"Type:          {body['type']}")
print(f"Sections used: {body['sections_used']}")
print(f"Torque specs:  {len(body['torque_specs'])}")
print(f"Image paths:   {len(body['image_paths'])}")
print()
print(body['text'])

## 2. Ambiguous query → disambiguation response

"Oil and filter change" matches sections in both Ch. 18 (engine-out assembly) and
Ch. 22 (routine maintenance).  The API returns a `"disambiguation"` response listing
the candidates grouped by chapter so the UI can present a picker.

In [ ]:
resp = client.post("/ask", json={"query": "oil and filter change"})
print(f"Status: {resp.status_code}")

body = resp.json()
print(f"Type: {body['type']}")
print(f"Query echoed: {body['query']}")
print()
print("Candidates:")
for group in body['candidates']:
    print(f"  Ch. {group['chapter_num']} — {group['chapter_title']}")
    for sec in group['sections']:
        print(f"    {sec['section']}  {sec['section_title'] or '(no title)'}")

## 3. POST /plan — scoped plan after disambiguation

The user selects Ch. 22.  The frontend calls `POST /plan` with the chosen section.
The response is a `"plan"` containing only Ch. 22 content — no engine-assembly
material from Ch. 18.

In [ ]:
resp = client.post("/plan", json={"query": "change the engine oil and filter", "section": "22.3"})
print(f"Status: {resp.status_code}")

body = resp.json()
print(f"Type:          {body['type']}")
print(f"Sections used: {body['sections_used']}")

chapters_in_plan = sorted({int(s.split('.')[0]) for s in body['sections_used']})
print(f"Chapters:      {chapters_in_plan}  (Ch. 18 excluded)")
print()
print(body['text'])

## 4. POST /plan — 404 for unknown section

In [ ]:
resp = client.post("/plan", json={"query": "test", "section": "99.99"})
print(f"Status: {resp.status_code}")
print(f"Detail: {resp.json()['detail']}")

## 5. GET /images — serve a diagram file

Image paths come back in every `PlanResponse`.  The frontend uses them to render
diagrams inline.  Here we fetch the first available image and verify the response.

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage

# Pull a real image path from the plan response
resp = client.post("/ask", json={"query": "adjust the handlebar position"})
image_paths = resp.json().get("image_paths", [])

if image_paths:
    # The stored path is relative to the project root (e.g. data/images/…)
    # The /images/ endpoint expects the part after data/images/
    relative = str(Path(image_paths[0]).relative_to("data/images"))
    img_resp = client.get(f"/images/{relative}")
    print(f"GET /images/{relative}")
    print(f"Status:       {img_resp.status_code}")
    print(f"Content-Type: {img_resp.headers.get('content-type')}")
    print(f"Size:         {len(img_resp.content)} bytes")

    # Display the image inline
    display(IPImage(data=img_resp.content, width=400))
else:
    print("No images in this result set.")

In [ ]:
# 404 for a missing path
resp = client.get("/images/nonexistent/file.jpg")
print(f"Status: {resp.status_code}")
print(f"Detail: {resp.json()['detail']}")

## 6. Response schema reference

FastAPI auto-generates an OpenAPI schema.  The full interactive docs are available
at `http://localhost:8000/docs` when the server is running.

Here we print the schema for both response types directly from the app.

In [ ]:
resp = client.get("/openapi.json")
schema = resp.json()

for name in ["PlanResponse", "DisambiguationResponse"]:
    defn = schema["components"]["schemas"][name]
    print(f"\n{name}:")
    for field, props in defn.get("properties", {}).items():
        typ = props.get("type") or props.get("$ref", "").split("/")[-1]
        print(f"  {field:<20} {typ}")